# Prepare summary dataframe for analysis

This notebook collects all the information needed for the gender bias analysis into a single dataframe and saves it to disk.

## Prerequisites

The following external resources are required:

- **MuST-C** (training corpus, target language side): [Cattoni et al., 2019](https://aclanthology.org/N19-1202/)
- **MuST-SHE** (evaluation benchmark): [Bentivogli et al., 2020](https://aclanthology.org/2020.acl-main.619/)
- **MuST-SHE POS extension**: [Bentivogli et al., 2022](https://aclanthology.org/2022.acl-long.127/)

The following files must be generated before running this notebook:

- `$gender_explanation_tsv`: model hypotheses on MuST-SHE annotated with gender term information — generated in **step 1** of [CONTRASTIVE_SPES.md](CONTRASTIVE_SPES.md)
- `$orig_probs`: token-level probabilities from the full ST model — generated in **step 2** of [CONTRASTIVE_SPES.md](CONTRASTIVE_SPES.md)
- `$ilm_probs`: token-level probabilities from the internal language model (encoder replaced with a zero vector) — generated with `examples/speech_to_text/scripts/xai/get_probs_from_ilm.py` (see repository README for details)
- `$deletion_output`: translation hypotheses at each spectrogram deletion step — generated in **step 4** of [CONTRASTIVE_SPES.md](CONTRASTIVE_SPES.md) with `--perc-interval 1` and `--max-percent 20`
- SPM vocabulary file for the target language — downloaded alongside the model checkpoint ([Transformer](https://github.com/facebookresearch/fairseq/blob/main/examples/speech_to_text/docs/mustc_example.md), [Conformer](https://github.com/hlt-mt/FBK-fairseq/blob/master/fbk_works/BUGFREE_CONFORMER.md))

**Output:** `summary_dataframe.tsv` — one row per gender term instance, with columns for model predictions, ILM probabilities, flip thresholds, and training data term frequencies.

## Configuration

Set the language and model, then replace the path placeholders with your own paths.

In [ ]:
LANG = "fr"          # one of: es, fr, it
MODEL = "conformer"  # one of: transformer, conformer

# Number of deletion steps: must match --perc-interval and --max-percent used
# in step 4 of CONTRASTIVE_SPES.md. With --perc-interval 1 --max-percent 20
# this is 21 (steps 0, 1, ..., 20).
NUM_DEL_STEPS_SRC = 21

# Input paths — replace placeholders with your own paths
explain_tsv              = "/path/to/gender_explanation_tsv.tsv"  # $gender_explanation_tsv from CONTRASTIVE_SPES.md
pos_tsv                  = "/path/to/MuST-SHE_POS_extension.tsv"
orig_probs_path          = "/path/to/orig_probs.h5"               # $orig_probs from CONTRASTIVE_SPES.md
ilm_probs_path           = "/path/to/ilm_probs.h5"                # $ilm_probs
src_deletion_output_path = "/path/to/deletion_output.out"         # $deletion_output from CONTRASTIVE_SPES.md
mustc_train_path         = "/path/to/mustc/train.LANG"
dict_path                = "/path/to/spm_vocabulary.txt"

# Output path
output_path = "/path/to/summary_dataframe.tsv"

## Imports

In [ ]:
import sys
import csv
from collections import defaultdict

import h5py
import numpy as np
import pandas as pd
import torch
from sacremoses import MosesTokenizer
from tqdm import tqdm

from examples.speech_to_text.occlusion_explanation.scorers.probability_aggregators import WordBoundaryProbabilityAggregator
from examples.speech_to_text.xai_metrics.auc_score import load_tsv, read_hypos
from examples.speech_to_text.xai_metrics.auc_score_mustshe import filter_samples
from examples.speech_to_text.xai_metrics.mustshe_scorers import GenderFlipRateScorer
from examples.speech_to_text.scripts.gender.mustshe_gender_accuracy import sentence_level_statistics
from fairseq.data.dictionary import Dictionary

## 1. Build the base dataframe

We merge the model's translation hypotheses (annotated with the gender term found and its target-sequence position) with the MuST-SHE POS annotations. We then derive a few convenience columns and drop columns that are not needed downstream.

In [ ]:
df_explain = pd.read_csv(explain_tsv, sep='\t', escapechar='\\', quoting=csv.QUOTE_NONE)
df_pos = pd.read_csv(pos_tsv, sep='\t')

# MuST-SHE POS file uses bare IDs; the hypothesis file appends '_0' to match segment-level IDs
df_pos['id'] = df_pos['ID'] + '_0'

df = df_explain.merge(df_pos, on='id', how='left')

In [ ]:
def get_gt_POS(row: pd.Series) -> str:
    """Return the POS tag for the specific gender term pair that was found in this example.

    Each MuST-SHE sentence can contain multiple gender term pairs (semicolon-separated);
    we select the POS tag corresponding to the one actually generated by the model.
    """
    pair_id = row['GENDERTERMS'].split(';').index(row['found_term_pairs'])
    return row['POS'].split(';')[pair_id]


def is_correct(row: pd.Series) -> bool:
    """Return True if the model generated the gender-correct form.

    In MuST-SHE, found_term_pairs stores 'correct_form incorrect_form' (space-separated),
    so we check that the generated term matches the first element.
    """
    return row['found_terms'] == row['found_term_pairs'].split(' ')[0]


df['pos']        = df.apply(get_gt_POS, axis=1)
df['correct']    = df.apply(is_correct, axis=1)
df['cat_nb']     = df['category'].str[0]  # category number (e.g. '1', '2')
df['cat_gender'] = df['category'].str[1]  # reference gender ('F' or 'M')

# Drop columns that are no longer needed
cols_to_drop = ['audio', 'n_frames', 'speaker', 'POS', 'GENDERTERMS', 'ID']
if MODEL == 'transformer':
    # The multilingual transformer TSV includes a tgt_lang column; monolingual models do not
    cols_to_drop.append('tgt_lang')
df.drop(columns=cols_to_drop, inplace=True)

## 2. Compute gender term probabilities

For each example we compute the probability assigned by the full model and the ILM to:
- the **predicted** form (whichever gender the model generated)
- the **swapped** form (the opposite-gender alternative)
- the **feminine** form (regardless of which was predicted)
- the **masculine** form (regardless of which was predicted)

Probabilities are read from pre-computed HDF5 files. The transformer is multilingual and prepends a language tag token to the target sequence, so its token indices are offset by 1 relative to the conformer models.

In [ ]:
tgt_dict = Dictionary.load(dict_path)
prob_aggregator = WordBoundaryProbabilityAggregator(tgt_dict)

orig_probs = h5py.File(orig_probs_path, 'r')
ilm_probs  = h5py.File(ilm_probs_path, 'r')
assert orig_probs.keys() == ilm_probs.keys(), "Mismatch between original and ILM probability files"

In [ ]:
# The transformer prepends a language tag token, so target token indices are shifted by 1
offset = 1 if MODEL == 'transformer' else 0
start_tgt_tokens = [tgt_dict.bos()] if MODEL == 'transformer' else []

term_probs = {key: {'orig': [], 'ilm': []} for key in ('predicted', 'swap', 'fem', 'masc')}

for i in range(len(orig_probs.keys()) // 2):
    start_idx = torch.tensor([int(df.gender_terms_indices.iloc[i].split('-')[0]) + offset])
    end_idx   = torch.tensor([int(df.gender_terms_indices.iloc[i].split('-')[1]) + offset])

    # The swapped translation may tokenize differently (e.g. 'diventato' vs 'diventata'),
    # so we adjust the end index by the difference in token count
    swapped_end_idx = end_idx + len(df.swapped_tgt_text.iloc[i].split(' ')) \
                              - len(df.tgt_text.iloc[i].split(' '))

    def get_prob(probs_file, key, tgt_text, start, end):
        return prob_aggregator.compute_word_probability(
            probs=torch.tensor(np.array(probs_file[key])).unsqueeze(0),
            tgt_tokens=torch.tensor(
                start_tgt_tokens
                + [tgt_dict.index(sym) for sym in tgt_text.split(' ')]
                + [tgt_dict.eos()]
            ).unsqueeze(0),
            start_indices=start,
            end_indices=end,
        ).item()

    orig_pred = get_prob(orig_probs, str(i),            df.tgt_text.iloc[i],         start_idx, end_idx)
    ilm_pred  = get_prob(ilm_probs,  str(i),            df.tgt_text.iloc[i],         start_idx, end_idx)

    # Swapped probabilities can fail if the swapped form contains OOV tokens
    try:
        orig_swap = get_prob(orig_probs, str(i) + '_swapped', df.swapped_tgt_text.iloc[i], start_idx, swapped_end_idx)
    except Exception:
        print(f"Warning: could not compute swapped probability for example {i}; setting to NaN.")
        orig_swap = np.nan
    ilm_swap = get_prob(ilm_probs, str(i) + '_swapped', df.swapped_tgt_text.iloc[i], start_idx, swapped_end_idx)

    term_probs['predicted']['orig'].append(orig_pred)
    term_probs['predicted']['ilm'].append(ilm_pred)
    term_probs['swap']['orig'].append(orig_swap)
    term_probs['swap']['ilm'].append(ilm_swap)

    # Map predicted/swapped to fem/masc based on the reference gender of this example.
    # For a feminine-reference example, the correct (feminine) form is the first in found_term_pairs.
    predicted_is_correct = (
        (df.category.iloc[i][-1] == 'F' and df.found_term_pairs.iloc[i].split(' ')[0] == df.found_terms.iloc[i])
        or
        (df.category.iloc[i][-1] == 'M' and df.found_term_pairs.iloc[i].split(' ')[1] == df.found_terms.iloc[i])
    )
    if predicted_is_correct:
        term_probs['fem']['orig'].append(orig_pred);  term_probs['fem']['ilm'].append(ilm_pred)
        term_probs['masc']['orig'].append(orig_swap); term_probs['masc']['ilm'].append(ilm_swap)
    else:
        term_probs['fem']['orig'].append(orig_swap);  term_probs['fem']['ilm'].append(ilm_swap)
        term_probs['masc']['orig'].append(orig_pred); term_probs['masc']['ilm'].append(ilm_pred)

orig_probs.close()
ilm_probs.close()

df['orig_prob_predicted'] = term_probs['predicted']['orig']
df['ilm_prob_predicted']  = term_probs['predicted']['ilm']
df['orig_prob_swap']      = term_probs['swap']['orig']
df['ilm_prob_swap']       = term_probs['swap']['ilm']
df['orig_prob_fem']       = term_probs['fem']['orig']
df['ilm_prob_fem']        = term_probs['fem']['ilm']
df['orig_prob_masc']      = term_probs['masc']['orig']
df['ilm_prob_masc']       = term_probs['masc']['ilm']

## 3. Compute flip thresholds

The contrastive feature attribution experiment progressively occludes the most salient spectrogram features and re-translates the audio. Here we record, for each example:
- `flip_percent`: the smallest percentage of features whose occlusion causes the model to generate the opposite-gender form
- `ooc_percent`: the smallest percentage at which the gender term is no longer generated at all (out of coverage)

`NaN` means the prediction never flipped within the 0–20% deletion range.

In [ ]:
hypos_dict = read_hypos(src_deletion_output_path)
hypos = hypos_dict[0][1]
mustshe_reference_list = load_tsv(explain_tsv)
step_to_hypos, step_to_refs = filter_samples(hypos, mustshe_reference_list, 0, sys.maxsize)

step_to_scores = {}
for step in step_to_hypos.keys():
    scorer = GenderFlipRateScorer()
    for r, h in zip(step_to_refs[step], step_to_hypos[step]):
        scorer.add_string(r, h)
    step_to_scores[step] = sentence_level_statistics(scorer.pred, scorer.mustshe)

# For each example, find the first deletion step at which the gender prediction flips
step_of_flip = [np.nan] * len(df)
for step in step_to_scores.keys():
    for i in range(len(df)):
        if step_to_scores[step][i]['num_correct'] == 1 and step_of_flip[i] is np.nan:
            step_of_flip[i] = step

# Also record when each example drops out of coverage (the gender term is no longer generated at all)
step_of_ooc = [np.nan] * len(df)
for step in sorted(step_to_scores.keys()):
    for i in range(len(df)):
        if step_to_scores[step][i]['num_terms_found'] == 0 and step_of_ooc[i] is np.nan:
            step_of_ooc[i] = step

df['flip_percent'] = step_of_flip
df['ooc_percent']  = step_of_ooc

## 4. Compute training data term frequencies

For each gender term pair in the dataset, we count how often each form appears in the MuST-C training corpus. We then record, for each example, the proportion of occurrences corresponding to the form the model predicted. A value near 1 means the model predicted the more frequent form; a value near 0 means it predicted the rarer form.

We use Moses tokenization to normalize terms before counting, to match the tokenization of the training data.

In [ ]:
mt = MosesTokenizer(lang=LANG)

with open(mustc_train_path, 'r') as f:
    tokenized_lines = [mt.tokenize(line.strip().lower()) for line in tqdm(f, desc='Tokenizing training data')]

# Count occurrences of each term form that appears in our evaluation data.
# We iterate over unique term pairs rather than individual terms to avoid redundant passes.
term_to_occurrences = defaultdict(int)
for term_pair in tqdm(df['found_term_pairs'].value_counts().index, desc='Counting term occurrences'):
    for term in term_pair.split(' '):
        normalized = mt.tokenize(term.lower())[0]
        if normalized not in term_to_occurrences:
            term_to_occurrences[normalized] = sum(1 for line in tokenized_lines if normalized in line)

In [ ]:
training_pref_for_predicted = []
for _, row in df.iterrows():
    # Identify which of the two forms was predicted and which was the alternative
    if row['found_terms'] == row['found_term_pairs'].split(' ')[0]:
        found_term = mt.tokenize(row['found_term_pairs'].split(' ')[0].lower())[0]
        other_term = mt.tokenize(row['found_term_pairs'].split(' ')[1].lower())[0]
    else:
        found_term = mt.tokenize(row['found_term_pairs'].split(' ')[1].lower())[0]
        other_term = mt.tokenize(row['found_term_pairs'].split(' ')[0].lower())[0]

    total = term_to_occurrences[found_term] + term_to_occurrences[other_term]
    training_pref_for_predicted.append(
        term_to_occurrences[found_term] / total if total > 0 else np.nan
    )

df['training_pref_for_predicted'] = training_pref_for_predicted

## 5. Save

In [ ]:
df.to_csv(output_path, sep='\t', index=False, escapechar='\\', quoting=csv.QUOTE_NONE)
print(f"Saved {len(df)} rows to {output_path}")
df.head(4)